# Day 11 — SQL Foundations Review + SQLite Setup
**Estimated time:** 45–60 min
**All 5 datasets loaded into SQLite in-memory DB**

## Learning Objectives
- Bootstrap a reproducible SQLite analytics environment from CSV data
- Review SELECT, WHERE, ORDER BY, LIMIT, DISTINCT, and CASE WHEN with SAP data
- Apply SQLite string functions to clean and categorize data inline

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

from analytics_bootcamp.config import RAW_DATA_DIR as DATA_DIR
# Load all CSVs
inv   = pd.read_csv(f"{DATA_DIR}/materials_inventory.csv")
sales = pd.read_csv(f"{DATA_DIR}/sales_orders.csv")
cc    = pd.read_csv(f"{DATA_DIR}/cost_center_actuals.csv")
hr    = pd.read_csv(f"{DATA_DIR}/hr_headcount.csv")
bw    = pd.read_csv(f"{DATA_DIR}/bw_sales_kpis.csv")

# Normalize column names
for df in [inv, sales, cc, hr, bw]:
    df.columns = [c.strip().upper() for c in df.columns]

# In-memory DuckDB — register pandas DataFrames as tables
con = duckdb.connect()
con.register("MATERIALS",   inv)
con.register("SALES",       sales)
con.register("COST_CENTER", cc)
con.register("HR",          hr)
con.register("BW_SALES",    bw)

def q(sql):
    return con.sql(sql).df()

# Verify
for tbl, df in [("MATERIALS",inv),("SALES",sales),("COST_CENTER",cc),("HR",hr),("BW_SALES",bw)]:
    n = q(f"SELECT COUNT(*) AS n FROM {tbl}").iloc[0,0]
    print(f"  {tbl:15s}: {n:,} rows")


In [ ]:
# -- SELECT + WHERE + ORDER BY + LIMIT --
result = q(
    """
    SELECT
        MATNR,
        MAKTX,
        WERKS,
        LABST,
        STPRS,
        LABST * STPRS AS INV_VALUE
    FROM MATERIALS
    WHERE LABST > 0
    ORDER BY INV_VALUE DESC
    LIMIT 20
    """
)
display(result)

In [ ]:
# -- DISTINCT with count per group --
result = q(
    """
    SELECT DISTINCT WERKS, COUNT(*) AS material_count
    FROM MATERIALS
    GROUP BY WERKS
    ORDER BY material_count DESC
    """
)
display(result)

In [ ]:
# -- CASE WHEN inventory value tier bucketing --
result = q(
    """
    SELECT
        MATNR,
        MAKTX,
        WERKS,
        LABST * STPRS AS INV_VALUE,
        CASE
            WHEN LABST = 0                               THEN 'Zero Stock'
            WHEN LABST * STPRS < 1000                    THEN '< $1K'
            WHEN LABST * STPRS BETWEEN 1000 AND 9999.99  THEN '$1K-$10K'
            ELSE '> $10K'
        END AS VALUE_TIER
    FROM MATERIALS
    ORDER BY INV_VALUE DESC
    """
)
print(result["VALUE_TIER"].value_counts())
display(result.head(20))

In [ ]:
# -- String functions: UPPER, TRIM, SUBSTR, INSTR --
result = q(
    """
    SELECT
        MATNR,
        MAKTX                           AS raw_desc,
        UPPER(TRIM(MAKTX))              AS desc_upper_trimmed,
        SUBSTR(MATNR, 1, 6)             AS matnr_prefix,
        LENGTH(TRIM(MAKTX))             AS desc_len,
        INSTR(UPPER(MAKTX), 'VALVE')    AS valve_position
    FROM MATERIALS
    WHERE INSTR(UPPER(MAKTX), 'VALVE') > 0
    LIMIT 10
    """
)
display(result)

In [ ]:
# -- CASE WHEN + GROUP BY: tier distribution --
result = q(
    """
    SELECT
        CASE
            WHEN LABST = 0                               THEN 'Zero Stock'
            WHEN LABST * STPRS < 1000                    THEN '< $1K'
            WHEN LABST * STPRS BETWEEN 1000 AND 9999.99  THEN '$1K-$10K'
            ELSE '> $10K'
        END AS VALUE_TIER,
        COUNT(DISTINCT MATNR) AS n_materials,
        SUM(LABST * STPRS)    AS total_inv_value
    FROM MATERIALS
    GROUP BY VALUE_TIER
    ORDER BY total_inv_value DESC
    """
)
display(result)

In [ ]:
# -- SALES overview by status --
result = q(
    """
    SELECT
        STATUS,
        COUNT(*)        AS order_count,
        SUM(NETWR)      AS total_revenue,
        AVG(NETWR)      AS avg_order_value,
        MIN(ERDAT)      AS earliest_order,
        MAX(ERDAT)      AS latest_order
    FROM SALES
    GROUP BY STATUS
    ORDER BY total_revenue DESC
    """
)
display(result)

---
## Daily Prompt

**Write a query that buckets materials into inventory value tiers and counts how many materials fall in each tier per plant (WERKS).**

Expected output columns: `WERKS`, `VALUE_TIER`, `n_materials`, `total_inv_value`

```sql
-- YOUR SOLUTION
SELECT
    WERKS,
    CASE
        WHEN LABST = 0              THEN 'Zero Stock'
        WHEN LABST * STPRS < 1000   THEN '< $1K'
        -- YOUR SOLUTION: complete the CASE WHEN tiers
    END AS VALUE_TIER,
    COUNT(DISTINCT MATNR)  AS n_materials,
    SUM(LABST * STPRS)     AS total_inv_value
FROM MATERIALS
GROUP BY WERKS, VALUE_TIER
ORDER BY WERKS, total_inv_value DESC
```

> **Hint:** SQLite has no native PIVOT. To see results in a wide format, read the query result into pandas and use `pivot_table()`.